# MS-GSF-BT-VAE — RTX 5060 실행본

이 노트북은 재설계한 `models1.py`를 사용합니다.

- Multi-scale residual GroupNorm Encoder
- Spatial variational bottleneck
- Uncertainty-aware gated skip Decoder
- Deep supervision
- BT projector 기반 자기지도 사전학습
- AMP, gradient accumulation, early stopping, 시간 제한

기존 `data_load_BT.py`와 이 노트북, 새 `models1.py`를 같은 폴더에 둔 뒤 커널을 재시작하세요.

In [ ]:
import os, time, json, random, csv
from pathlib import Path
from contextlib import nullcontext

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from data_load_BT import datasets
from models1 import Encoder, VAE

SEED = 42
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"

# 모델 구조: 메모리가 부족해도 아래 세 값은 유지하고 batch만 줄입니다.
BASE = 64
LATENT_CH = 1024
DROPOUT = 0.05

BT_BATCH = 8
BT_ACCUM = 4
SEG_BATCH = 4
SEG_ACCUM = 4
NUM_WORKERS = 0  # Windows resume/spawn 안정성

BT_EPOCHS = 400
SEG_EPOCHS = 400
BT_PATIENCE = 40
SEG_PATIENCE = 50
PRETRAIN_MAX_HOURS = 96.0
MAX_TOTAL_HOURS = 240.0

ROOT = Path(".")
CKPT_DIR = ROOT / "checkpoints"
RESULT_DIR = ROOT / "results" / "ms_gsf_bt_vae"
CKPT_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)
BT_BEST = CKPT_DIR / "ms_gsf_bt_encoder_best.pt"
BT_LAST = CKPT_DIR / "ms_gsf_bt_encoder_last.pt"
SEG_BEST = RESULT_DIR / "seg_best.pt"
SEG_LAST = RESULT_DIR / "seg_last.pt"
RUN_START = time.time()

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

seed_everything(SEED)
print("device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

## 데이터 로더

OOM 발생 시 모델 구조는 그대로 두고 `BT_BATCH=4`, `BT_ACCUM=8`, `SEG_BATCH=2`, `SEG_ACCUM=8`로 변경합니다.

In [ ]:
bt_data = datasets.covid(batch_size=BT_BATCH, num_workers=NUM_WORKERS)
bt_train_loader = bt_data.train_loader
bt_val_loader = bt_data.val_loader

seg_data = datasets.covid(batch_size=SEG_BATCH, num_workers=NUM_WORKERS)
seg_train_loader = seg_data.train_loader
seg_val_loader = seg_data.val_loader
seg_test_loader = seg_data.test_loader

print("BT:", len(bt_data.train), len(bt_data.val), len(bt_data.test))
print("SEG:", len(seg_data.train), len(seg_data.val), len(seg_data.test))

## 모델 smoke test

256×256 입력에 대해 segmentation 출력도 256×256이어야 합니다. 이 셀이 실패하면 학습을 시작하지 마세요.

In [ ]:
_test_model = VAE(base=BASE, dropout_rate=DROPOUT, latent_ch=LATENT_CH).to(DEVICE)
_test_model.eval()
with torch.no_grad():
    _x = torch.randn(1, 1, 256, 256, device=DEVICE)
    _out, _mu, _lv, _aux = _test_model(_x, sample=False, return_aux=True)
print("out:", tuple(_out.shape), "mu:", tuple(_mu.shape), "aux:", [tuple(a.shape) for a in _aux])
print(f"parameters: {sum(p.numel() for p in _test_model.parameters())/1e6:.2f} M")
del _test_model, _x, _out, _mu, _lv, _aux
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

## 공통 함수

In [ ]:
def as_bchw(x):
    x = x.float()
    if x.ndim == 3:
        x = x.unsqueeze(1)
    if x.ndim != 4:
        raise ValueError(f"Expected BCHW/BHW, got {tuple(x.shape)}")
    return x


def make_scaler():
    if hasattr(torch, "amp") and hasattr(torch.amp, "GradScaler"):
        try:
            return torch.amp.GradScaler("cuda", enabled=USE_AMP)
        except TypeError:
            pass
    return torch.cuda.amp.GradScaler(enabled=USE_AMP)


def amp_context():
    if not USE_AMP:
        return nullcontext()
    if hasattr(torch, "amp") and hasattr(torch.amp, "autocast"):
        return torch.amp.autocast("cuda", dtype=torch.float16)
    return torch.cuda.amp.autocast(dtype=torch.float16)


def off_diagonal(x):
    n, m = x.shape
    if n != m:
        raise ValueError("cross-correlation matrix must be square")
    return x.flatten()[:-1].view(n - 1, n + 1)[:, 1:].flatten()


def barlow_loss(z1, z2, lambd=5e-3, eps=1e-4):
    z1 = (z1 - z1.mean(0)) / (z1.std(0, unbiased=False) + eps)
    z2 = (z2 - z2.mean(0)) / (z2.std(0, unbiased=False) + eps)
    c = z1.T @ z2 / z1.size(0)
    on = (torch.diagonal(c) - 1).pow(2).mean()
    off = off_diagonal(c).pow(2).mean()
    return on + lambd * off


def symmetric_diag_kl(mu1, lv1, mu2, lv2):
    mu1 = F.adaptive_avg_pool2d(mu1, 1).flatten(1).float()
    mu2 = F.adaptive_avg_pool2d(mu2, 1).flatten(1).float()
    lv1 = F.adaptive_avg_pool2d(lv1, 1).flatten(1).float().clamp(-6, 2)
    lv2 = F.adaptive_avg_pool2d(lv2, 1).flatten(1).float().clamp(-6, 2)
    v1, v2 = lv1.exp(), lv2.exp()
    d2 = (mu1 - mu2).pow(2)
    kl12 = 0.5 * (lv2 - lv1 + (v1 + d2) / v2 - 1)
    kl21 = 0.5 * (lv1 - lv2 + (v2 + d2) / v1 - 1)
    return 0.5 * (kl12.mean() + kl21.mean())


def time_exceeded(hours):
    return (time.time() - RUN_START) >= hours * 3600


# ---------- interrupt-safe checkpoint helpers ----------
def atomic_torch_save(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)


def capture_rng():
    state = {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch": torch.get_rng_state(),
    }
    if torch.cuda.is_available():
        state["cuda"] = torch.cuda.get_rng_state_all()
    return state


def restore_rng(state):
    if not state:
        return
    try:
        random.setstate(state["python"])
        np.random.set_state(state["numpy"])
        torch.set_rng_state(state["torch"])
        if torch.cuda.is_available() and "cuda" in state:
            torch.cuda.set_rng_state_all(state["cuda"])
    except Exception as e:
        print("[warn] RNG restore skipped:", e)


def safe_load(path, map_location=DEVICE):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)


def check_config(saved, where):
    if not saved:
        return
    cur = {"base": BASE, "latent_ch": LATENT_CH, "dropout": DROPOUT}
    for k, v in cur.items():
        if k in saved and saved[k] != v:
            raise RuntimeError(f"Config mismatch in {where}: saved {k}={saved[k]} vs {v}")


## BT 사전학습

Projection representation에는 Barlow Twins loss를 적용하고, spatial Gaussian latent에는 동일 채널 간 symmetric KL consistency만 적용합니다. 기존 1024×1024 KL 행렬보다 메모리 부담이 작습니다.

In [ ]:
def run_bt_epoch(encoder, loader, optimizer=None, scaler=None, accum_steps=1):
    training = optimizer is not None
    encoder.train(training)
    total, total_bt, total_dist = 0.0, 0.0, 0.0
    if training:
        optimizer.zero_grad(set_to_none=True)

    for step, batch in enumerate(loader, 1):
        v1 = as_bchw(batch[2]).to(DEVICE, non_blocking=True)
        v2 = as_bchw(batch[3]).to(DEVICE, non_blocking=True)
        ctx = amp_context() if training else torch.no_grad()
        with ctx:
            p1, mu1, lv1 = encoder.forward_bt(v1)
            p2, mu2, lv2 = encoder.forward_bt(v2)
            loss_bt = barlow_loss(p1.float(), p2.float())
            loss_dist = symmetric_diag_kl(mu1, lv1, mu2, lv2)
            loss = loss_bt + 0.05 * loss_dist

        if training:
            scaler.scale(loss / accum_steps).backward()
            if step % accum_steps == 0 or step == len(loader):
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(encoder.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

        total += loss.detach().item()
        total_bt += loss_bt.detach().item()
        total_dist += loss_dist.detach().item()
    n = max(1, len(loader))
    return {"loss": total/n, "bt": total_bt/n, "dist": total_dist/n}


def save_bt_state(encoder, optimizer, scheduler, epoch, best_bt, bad_epochs):
    state = {
        "phase": "bt",
        "epoch": epoch,
        "encoder": encoder.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "best": best_bt,
        "bad_epochs": bad_epochs,
        "config": {"base": BASE, "latent_ch": LATENT_CH, "dropout": DROPOUT},
        "rng": capture_rng(),
    }
    atomic_torch_save(state, BT_LAST)
    return state


encoder = Encoder(BASE, DROPOUT, LATENT_CH).to(DEVICE)
bt_optimizer = torch.optim.AdamW(encoder.parameters(), lr=2e-4, weight_decay=1e-4)
bt_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    bt_optimizer, mode="min", factor=0.5, patience=5, min_lr=1e-6
)
bt_scaler = make_scaler()
best_bt, bad_epochs = float("inf"), 0
start_epoch = 1

if BT_LAST.exists():
    state = safe_load(BT_LAST)
    check_config(state.get("config"), str(BT_LAST))
    encoder.load_state_dict(state["encoder"])
    bt_optimizer.load_state_dict(state["optimizer"])
    if "scheduler" in state:
        bt_scheduler.load_state_dict(state["scheduler"])
    start_epoch = int(state["epoch"]) + 1
    best_bt = float(state.get("best", best_bt))
    bad_epochs = int(state.get("bad_epochs", 0))
    restore_rng(state.get("rng"))
    print(f"BT resume: epoch={start_epoch}, best={best_bt:.5f}, bad={bad_epochs}")

bt_phase_start = time.time()
try:
    for epoch in range(start_epoch, BT_EPOCHS + 1):
        train_m = run_bt_epoch(encoder, bt_train_loader, bt_optimizer, bt_scaler, BT_ACCUM)
        val_m = run_bt_epoch(encoder, bt_val_loader)
        bt_scheduler.step(val_m["loss"])

        if val_m["loss"] < best_bt:
            best_bt, bad_epochs = val_m["loss"], 0
            st = save_bt_state(encoder, bt_optimizer, bt_scheduler, epoch, best_bt, bad_epochs)
            st["best"] = best_bt
            atomic_torch_save(st, BT_BEST)
        else:
            bad_epochs += 1
            save_bt_state(encoder, bt_optimizer, bt_scheduler, epoch, best_bt, bad_epochs)

        print(f"[BT {epoch:03d}] train={train_m['loss']:.5f} val={val_m['loss']:.5f} "
              f"bt={val_m['bt']:.5f} dist={val_m['dist']:.5f} bad={bad_epochs}")
        if bad_epochs >= BT_PATIENCE:
            print("BT early stopping")
            break
        if time.time() - bt_phase_start >= PRETRAIN_MAX_HOURS * 3600:
            print("BT time budget reached")
            break
except KeyboardInterrupt:
    print("\n[BT] interrupted — checkpoint saved to BT_LAST; re-run this cell to resume")
    save_bt_state(encoder, bt_optimizer, bt_scheduler, max(start_epoch - 1, 1), best_bt, bad_epochs)


## Segmentation 손실과 지표

구조 효과를 먼저 검증하기 위해 main loss는 기존과 같은 Dice+BCE를 사용합니다. Auxiliary head는 학습 때만 사용합니다.

In [ ]:
def dice_bce(logits, target):
    probs = torch.sigmoid(logits)
    dims = tuple(range(1, probs.ndim))
    inter = (probs * target).sum(dims)
    denom = probs.sum(dims) + target.sum(dims)
    dice_loss = 1 - ((2 * inter + 1e-6) / (denom + 1e-6)).mean()
    bce = F.binary_cross_entropy_with_logits(logits, target)
    return 0.5 * dice_loss + 0.5 * bce


def segmentation_loss(logits, target, mu, logvar, aux, epoch):
    main = dice_bce(logits, target)
    aux_loss = 0.3 * dice_bce(aux[0], target) + 0.15 * dice_bce(aux[1], target)
    kl = (-0.5 * (1 + logvar - mu.pow(2) - logvar.exp())).flatten(1).mean()
    beta = 1e-3 * min(1.0, epoch / 25.0)
    return main + aux_loss + beta * kl, main.detach(), kl.detach()


@torch.no_grad()
def batch_metrics(logits, target, threshold=0.5):
    pred = (torch.sigmoid(logits) >= threshold).float()
    target = (target >= 0.5).float()
    dims = tuple(range(1, pred.ndim))
    inter = (pred * target).sum(dims)
    pred_sum, target_sum = pred.sum(dims), target.sum(dims)
    union = pred_sum + target_sum - inter
    dice = ((2 * inter + 1e-6) / (pred_sum + target_sum + 1e-6)).mean()
    iou = ((inter + 1e-6) / (union + 1e-6)).mean()
    return dice.item(), iou.item()


def run_seg_epoch(model, loader, epoch, optimizer=None, scaler=None, accum_steps=1):
    training = optimizer is not None
    model.train(training)
    sums = {"loss": 0.0, "main": 0.0, "kl": 0.0, "dice": 0.0, "iou": 0.0}
    if training:
        optimizer.zero_grad(set_to_none=True)

    for step, batch in enumerate(loader, 1):
        x = as_bchw(batch[0]).to(DEVICE, non_blocking=True)
        y = as_bchw(batch[1]).to(DEVICE, non_blocking=True)
        ctx = amp_context() if training else torch.no_grad()
        with ctx:
            logits, mu, logvar, aux = model(x, sample=training, return_aux=True)
            loss, main, kl = segmentation_loss(logits, y, mu, logvar, aux, epoch)

        if training:
            scaler.scale(loss / accum_steps).backward()
            if step % accum_steps == 0 or step == len(loader):
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

        dice, iou = batch_metrics(logits.detach(), y)
        sums["loss"] += loss.detach().item()
        sums["main"] += main.item()
        sums["kl"] += kl.item()
        sums["dice"] += dice
        sums["iou"] += iou

    n = max(1, len(loader))
    return {k: v/n for k, v in sums.items()}

## Fine-tuning

검증은 확률 샘플링 대신 `z=mu`를 사용해 결정적으로 계산합니다. Scheduler는 epoch당 한 번만 호출합니다.

In [ ]:
vae = VAE(BASE, DROPOUT, LATENT_CH).to(DEVICE)
seg_optimizer = torch.optim.AdamW(vae.parameters(), lr=1e-4, weight_decay=1e-4)
seg_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    seg_optimizer, mode="min", factor=0.5, patience=7, min_lr=1e-7
)
seg_scaler = make_scaler()
best_val, bad_epochs, history = float("inf"), 0, []
start_epoch = 1

if SEG_LAST.exists():
    state = safe_load(SEG_LAST)
    check_config(state.get("config"), str(SEG_LAST))
    vae.load_state_dict(state["model"])
    seg_optimizer.load_state_dict(state["optimizer"])
    if "scheduler" in state:
        seg_scheduler.load_state_dict(state["scheduler"])
    start_epoch = int(state["epoch"]) + 1
    best_val = float(state.get("best", best_val))
    bad_epochs = int(state.get("bad_epochs", 0))
    history = state.get("history", [])
    restore_rng(state.get("rng"))
    print(f"SEG resume: epoch={start_epoch}, best={best_val:.5f}, bad={bad_epochs}")
else:
    if not BT_BEST.exists():
        raise FileNotFoundError(f"BT checkpoint not found: {BT_BEST}")
    bt_state = safe_load(BT_BEST)
    check_config(bt_state.get("config"), str(BT_BEST))
    missing, unexpected = vae.encoder.load_state_dict(bt_state["encoder"], strict=True)
    print("encoder loaded; missing:", missing, "unexpected:", unexpected)


def save_seg_state(epoch, best_val, bad_epochs, history):
    state = {
        "phase": "seg",
        "epoch": epoch,
        "model": vae.state_dict(),
        "optimizer": seg_optimizer.state_dict(),
        "scheduler": seg_scheduler.state_dict(),
        "best": best_val,
        "bad_epochs": bad_epochs,
        "history": history,
        "config": {"base": BASE, "latent_ch": LATENT_CH, "dropout": DROPOUT},
        "rng": capture_rng(),
    }
    atomic_torch_save(state, SEG_LAST)
    return state

try:
    for epoch in range(start_epoch, SEG_EPOCHS + 1):
        tr = run_seg_epoch(vae, seg_train_loader, epoch, seg_optimizer, seg_scaler, SEG_ACCUM)
        va = run_seg_epoch(vae, seg_val_loader, epoch)
        seg_scheduler.step(va["loss"])
        row = {"epoch": epoch, **{f"train_{k}": v for k, v in tr.items()},
               **{f"val_{k}": v for k, v in va.items()},
               "lr": seg_optimizer.param_groups[0]["lr"]}
        history.append(row)

        if va["loss"] < best_val:
            best_val, bad_epochs = va["loss"], 0
            st = save_seg_state(epoch, best_val, bad_epochs, history)
            st["best"] = best_val
            atomic_torch_save(st, SEG_BEST)
        else:
            bad_epochs += 1
            save_seg_state(epoch, best_val, bad_epochs, history)

        with open(RESULT_DIR / "history.csv", "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=history[0].keys())
            writer.writeheader()
            writer.writerows(history)

        print(f"[SEG {epoch:03d}] train loss={tr['loss']:.5f} D={tr['dice']:.4f} IoU={tr['iou']:.4f} | "
              f"val loss={va['loss']:.5f} D={va['dice']:.4f} IoU={va['iou']:.4f} bad={bad_epochs}")
        if bad_epochs >= SEG_PATIENCE:
            print("SEG early stopping")
            break
        if time_exceeded(MAX_TOTAL_HOURS):
            print("Total time budget reached")
            break
except KeyboardInterrupt:
    print("\n[SEG] interrupted — checkpoint saved to SEG_LAST; re-run this cell to resume")
    save_seg_state(max(start_epoch - 1, 1), best_val, bad_epochs, history)


## Test 평가

In [ ]:
state = torch.load(SEG_BEST, map_location=DEVICE)
vae.load_state_dict(state["model"])
test_m = run_seg_epoch(vae, seg_test_loader, epoch=SEG_EPOCHS)
print("TEST:", test_m)
with open(RESULT_DIR / "test_metrics.json", "w", encoding="utf-8") as f:
    json.dump(test_m, f, indent=2)

## OOM 대응

1. `BT_BATCH=4`, `BT_ACCUM=8`로 변경합니다.
2. `SEG_BATCH=2`, `SEG_ACCUM=8`로 변경합니다.
3. 커널을 재시작하고 처음부터 실행합니다.
4. 모델의 `BASE`, `LATENT_CH`, 입력 크기는 논문 비교의 공정성을 위해 바꾸지 않습니다.

`torch.cuda.empty_cache()`는 사용하지 않는 캐시를 반환할 뿐 PyTorch 자체에 더 많은 학습 메모리를 만들어 주지는 않으므로, OOM 해결의 핵심은 micro-batch 축소와 AMP입니다.